<a href="https://colab.research.google.com/github/Projit1101/Financial-fraud-classification/blob/main/notebooks/graphsage_xgboost_ensemble.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Check if GPU is available

In [ ]:
import torch
print(torch.cuda.is_available())

True


Define the device (Output should be: cuda)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


Run this once per session (Access to my Google Drive)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Cell 1: Install Dependencies
# We can rely on Colab's pre-installed PyTorch and just install torch-geometric directly.
!pip install torch-geometric scikit-learn matplotlib pandas
!pip install faiss-cpu
!pip install torch-geometric scikit-learn pandas xgboost
!pip install pyg_lib torch_sparse torch_scatter -f https://data.pyg.org/whl/torch-$(python -c "import torch; print(torch.__version__)").html

Looking in links: https://data.pyg.org/whl/torch-2.10.0+cu128.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 31.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 138.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 128.1 MB/s eta 0:00:00


In [ ]:
# Cell 1:

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load Data
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Datasets/creditcard.csv')
features = df.drop(columns=['Time', 'Class']).values
labels = df['Class'].values

scaler = StandardScaler()
features = scaler.fit_transform(features)

# Recreate the exact same splits
train_idx, temp_idx = train_test_split(
    np.arange(features.shape[0]), test_size=0.3, random_state=42, stratify=labels
)
val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.5, random_state=42, stratify=labels[temp_idx]
)

X_train, y_train = features[train_idx], labels[train_idx]
X_val, y_val = features[val_idx], labels[val_idx]
X_test, y_test = features[test_idx], labels[test_idx]

In [ ]:
# Cell 2:

import xgboost as xgb

# Calculate ratio for scale_pos_weight to handle extreme imbalance
ratio = float(np.sum(y_train == 0)) / np.sum(y_train == 1)

xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=4,
    scale_pos_weight=ratio,
    random_state=42,
    eval_metric='aucpr',
    early_stopping_rounds=10  # Moved here
)

xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

# Get tabular probabilities for Val and Test
xgb_val_probs = xgb_model.predict_proba(X_val)[:, 1]
xgb_test_probs = xgb_model.predict_proba(X_test)[:, 1]

~ 3 minutes

In [ ]:
# Cell 3:

import faiss
import torch
from torch_geometric.data import Data
from torch_geometric.loader import NeighborLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Create Global Tensors
X_tensor = torch.tensor(features, dtype=torch.float32).to(device)
y_tensor = torch.tensor(labels, dtype=torch.long).to(device)

# 2. Build FAISS Capped Radius Graph
def create_capped_radius_edge_index(features_np, train_indices, max_neighbors=100, radius=3.0):
    d = features_np.shape[1]
    train_features = np.ascontiguousarray(features_np[train_indices], dtype=np.float32)
    index = faiss.IndexFlatL2(d)
    index.add(train_features)

    all_features = np.ascontiguousarray(features_np, dtype=np.float32)
    distances, local_neighbor_indices = index.search(all_features, max_neighbors)

    radius_squared = radius ** 2
    valid_mask = (distances <= radius_squared) & (local_neighbor_indices != -1)

    rows = np.repeat(np.arange(features_np.shape[0]), max_neighbors)
    cols = train_indices[local_neighbor_indices.flatten()]

    flat_mask = valid_mask.flatten()
    rows = rows[flat_mask]
    cols = cols[flat_mask]

    self_loop_mask = rows != cols
    rows = rows[self_loop_mask]
    cols = cols[self_loop_mask]

    return torch.tensor(np.vstack((cols, rows)), dtype=torch.long).to(device)

print("Building FAISS Graph...")
edge_index = create_capped_radius_edge_index(features, train_idx, max_neighbors=100, radius=3.0)

# 3. Create PyG Data Object
train_mask = torch.zeros(features.shape[0], dtype=torch.bool).to(device)
train_mask[train_idx] = True
val_mask = torch.zeros(features.shape[0], dtype=torch.bool).to(device)
val_mask[val_idx] = True
test_mask = torch.zeros(features.shape[0], dtype=torch.bool).to(device)
test_mask[test_idx] = True

data = Data(x=X_tensor, edge_index=edge_index, y=y_tensor,
            train_mask=train_mask, val_mask=val_mask, test_mask=test_mask)

# 4. Initialize NeighborLoaders
print("Creating NeighborLoaders...")
val_loader = NeighborLoader(data, num_neighbors=[15, 10], batch_size=2048, input_nodes=data.val_mask)
test_loader = NeighborLoader(data, num_neighbors=[15, 10], batch_size=2048, input_nodes=data.test_mask)
print("Ready for Step 3.")

Building FAISS Graph...
Creating NeighborLoaders...
Ready for Step 3.


In [ ]:
# Cell 4:

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv
from google.colab import files

# 1. Redefine the Model Architecture
class GraphSAGE_FraudModel(nn.Module):
    def __init__(self, in_features, hidden_dim, out_features, dropout_rate=0.5):
        super(GraphSAGE_FraudModel, self).__init__()
        self.conv1 = SAGEConv(in_features, hidden_dim)
        self.conv2 = SAGEConv(hidden_dim, out_features)
        self.dropout_rate = dropout_rate

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = self.conv1(x, edge_index)
        x = F.elu(x)
        x = F.dropout(x, p=self.dropout_rate, training=self.training)
        x = self.conv2(x, edge_index)
        return F.log_softmax(x, dim=1)

# Initialize model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_features = X_train.shape[1]
num_classes = 2

gnn_model = GraphSAGE_FraudModel(in_features=num_features, hidden_dim=16, out_features=num_classes).to(device)

# Prompt the user to upload the file
print("Please upload your saved model weights (.pth file)...")
uploaded = files.upload()

# Extract the filename from the upload dictionary
model_filename = list(uploaded.keys())[0]

# Load the saved state dict
gnn_model.load_state_dict(torch.load(model_filename, map_location=device))
gnn_model.eval()

# 2. Define the extraction function
def get_loader_predictions(loader, model):
    all_probs = []

    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            out = model(batch)
            # Extract probability for the positive class (fraud)
            probs = torch.exp(out[:batch.batch_size, 1])
            all_probs.append(probs.cpu())

    return torch.cat(all_probs).numpy()

# 3. Extract the GraphSAGE probabilities
print("Extracting GraphSAGE Validation Probabilities...")
gnn_val_probs = get_loader_predictions(val_loader, gnn_model)

print("Extracting GraphSAGE Test Probabilities...")
gnn_test_probs = get_loader_predictions(test_loader, gnn_model)

print("Extraction complete.")

Please upload your saved model weights (.pth file)...


Saving graphSAGE_radius_search.pth to graphSAGE_radius_search (1).pth
Extracting GraphSAGE Validation Probabilities...
Extracting GraphSAGE Test Probabilities...
Extraction complete.


In [ ]:
# Cell 5:

from sklearn.metrics import f1_score, roc_auc_score, accuracy_score, precision_score, recall_score, precision_recall_curve
import numpy as np

print("--- Tuning Weighted Average on Validation Set ---")

best_weight = 0
best_threshold = 0
best_val_f1 = 0

# Test XGBoost weights from 0.0 to 1.0 (in increments of 0.05)
for xgb_weight in np.arange(0.0, 1.05, 0.05):
    gnn_weight = 1.0 - xgb_weight

    # Calculate weighted probabilities
    blended_val_probs = (xgb_val_probs * xgb_weight) + (gnn_val_probs * gnn_weight)

    # Find optimal threshold for this specific weight combination
    precisions, recalls, thresholds = precision_recall_curve(y_val, blended_val_probs)
    f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)

    idx = np.argmax(f1_scores)
    max_f1 = f1_scores[idx]

    if max_f1 > best_val_f1:
        best_val_f1 = max_f1
        best_weight = xgb_weight
        best_threshold = thresholds[idx]

print(f"Optimal XGBoost Weight: {best_weight:.2f}")
print(f"Optimal GraphSAGE Weight: {1.0 - best_weight:.2f}")
print(f"Optimal Threshold: {best_threshold:.4f}")
print(f"Validation F1 Score: {best_val_f1:.4f}")

# --- Evaluate on Unseen Test Set ---
# Apply the best weights and threshold to the test set
blended_test_probs = (xgb_test_probs * best_weight) + (gnn_test_probs * (1.0 - best_weight))
y_pred_test = (blended_test_probs >= best_threshold).astype(int)

print("\n--- Final Weighted Ensemble Test Set Results ---")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_test):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_test, zero_division=1):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_test, zero_division=1):.4f}")
print(f"F1 Score:  {f1_score(y_test, y_pred_test, zero_division=1):.4f}")
print(f"AUC-ROC:   {roc_auc_score(y_test, blended_test_probs):.4f}")

--- Tuning Weighted Average on Validation Set ---
Optimal XGBoost Weight: 0.85
Optimal GraphSAGE Weight: 0.15
Optimal Threshold: 0.8511
Validation F1 Score: 0.7770

--- Final Weighted Ensemble Test Set Results ---
Accuracy:  0.9993
Precision: 0.8833
Recall:    0.7162
F1 Score:  0.7910
AUC-ROC:   0.9497


In [ ]:
import torch
from google.colab import files

# Define the file name
model_save_path = 'graphsage_xgboost_ensemble.pth'

# Save the model's learned weights (state dictionary)
torch.save(gnn_model.state_dict(), model_save_path)
print(f"Model successfully saved to Colab environment as: {model_save_path}")

# Download the file to your local computer for permanent safekeeping
print("Downloading to your local machine...")
files.download(model_save_path)

Model successfully saved to Colab environment as: graphsage_xgboost_ensemble.pth


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>